# TD-MAD v2 — pilot cổng G1 trên Kaggle (GPU T4 x2)

**Cài đặt notebook:** Accelerator = *GPU T4 x2*, Internet = *On*, Persistence = *Files only*.
**Input:** thêm dataset chứa thư mục `TD-MAD` (đã có sẵn `data/`), ví dụ `td-mad`.
Phiên sau: thêm output của phiên trước làm input để chạy tiếp (mọi bước đều resume).

Chạy nền không cần mở trình duyệt: *Save Version → Save & Run All (Commit)*. Phiên tối đa 12 giờ;
driver tự dừng nhận việc mới sau 11,3 giờ (`TDMAD_DEADLINE_TS`).

Thứ tự phiên 1: setup → test CPU → bench (10 phút) → smoke (20–30 phút) → 2 lane song song → báo cáo.

In [ ]:
import os, glob, shutil, subprocess, time
FIRST_SESSION = True    # bench + smoke only in the first session; set False when resuming
T0 = time.time()
os.environ["TDMAD_DEADLINE_TS"] = str(T0 + 11.3 * 3600)   # stop taking new work before the 12 h limit

SRC = None
for p in glob.glob("/kaggle/input/**/tdmad/__init__.py", recursive=True):
    SRC = os.path.dirname(os.path.dirname(p)); break
assert SRC, "add the TD-MAD dataset as input"
WORK = "/kaggle/working/TD-MAD"
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns("results", "logs", "__pycache__"))
os.chdir(WORK)
print("code from", SRC, "->", WORK)

In [ ]:
# optional: restore results of an earlier session (its output added as input)
for d in glob.glob("/kaggle/input/**/TD-MAD/results", recursive=True):
    if not d.startswith(SRC):
        shutil.copytree(d, f"{WORK}/results", dirs_exist_ok=True); print("restored", d); break

In [ ]:
# vLLM: the first session installs the latest release and the drivers record its version in
# results/<run>/meta/vllm_version.txt; every later session installs that same version
VLLM_PIN = None
for f in sorted(glob.glob(f"{WORK}/results/*/meta/vllm_version.txt")):
    VLLM_PIN = open(f).read().strip(); break
spec = f"vllm=={VLLM_PIN}" if VLLM_PIN else "vllm"
print("installing", spec)
!pip install -q {spec} 2>&1 | tail -n 2
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.device_count(), 'GPU')"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv


In [ ]:
# CPU tests (~10 s): parsing, graphs, traps, fake debate, client against a mock server
!python -m tests.test_core && python -m tests.test_client

In [ ]:
# first session only: measured throughput of both pool models (~10 min) and the calibrated budget
def sh(cmd, n=40):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("
".join((p.stdout + p.stderr).splitlines()[-n:]))
    return p.returncode

if FIRST_SESSION:
    sh("bash kaggle/run_g1.sh bench")

In [ ]:
# first session only: 3 items per task through S1-S4 on both lanes (configs/g1_smoke.yaml)
if FIRST_SESSION:
    rc = sh("bash kaggle/run_g1.sh smoke", n=30)
    print("smoke exit code", rc, "- read logs/smoke_lane*.log if not 0")

In [ ]:
# main run: lane0 (GPU0) and lane1 (GPU1) in parallel; logs in logs/lane*.log
os.makedirs("logs", exist_ok=True)
procs = {}
for lane in ("lane0", "lane1"):
    log = open(f"logs/{lane}.log", "a")
    procs[lane] = subprocess.Popen(["bash", "kaggle/run_g1.sh", lane], stdout=log, stderr=subprocess.STDOUT)

def tail(path, n=2):
    try:
        return open(path, errors="replace").read().splitlines()[-n:]
    except FileNotFoundError:
        return []

while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    print(f"--- {(time.time() - T0) / 3600:.1f} h")
    for lane in procs:
        print(lane, procs[lane].poll(), tail(f"logs/{lane}.log"))
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout)
print({k: p.returncode for k, p in procs.items()})

In [ ]:
# gate report (criterion 3 stays PENDING until the two annotators fill results/<run>/annotation/*.csv)
!python -m analysis.g1_gate --config configs/g1.yaml
!python -m analysis.cost_report --config configs/g1.yaml

In [ ]:
!bash kaggle/start_vllm.sh stop
!du -sh results; ls results/*/